# Period input

**What's in this notebook?** This notebook demonstrates how to supply custom period vectors or prepotentials directly to JAXVacua, without loading a pre-computed geometry. This is useful for one-modulus models, K-point / C-point prepotentials, or any compactification where the periods are known analytically.

**In this notebook, you will learn:**
- The calling conventions for `prepotential_input` and `period_input`
- How the conjugation flag `conj` works and when you need to implement it explicitly
- How to validate period normalisation using the symplectic pairing $i\,\Pi^\dagger \Sigma\,\Pi > 0$
- How to build a working `FluxEFT` model from a custom prepotential and run a full vacuum search

**Prerequisites:** Basic JAXVacua usage ([notebook 2](../01_basics/2_jaxvacua_overview.ipynb)), finding flux vacua ([notebook 5](../02_vacuum_finding/05_finding_flux_vacua.ipynb)).

**Related notebooks:** [05_finding_flux_vacua](../02_vacuum_finding/05_finding_flux_vacua.ipynb) (Newton search workflow), [15_hypergeometric_models](15_hypergeometric_models.ipynb) (built-in one-modulus models via closed-form prepotentials).

(*Created:* Andreas Schachner, June 25, 2024)

## Imports

In [ ]:
# General imports
import warnings
import numpy as np
from tqdm.auto import tqdm
from functools import partial

# JAX imports
from jax import jit
import jax
import jax.numpy as jnp
jax.config.update("jax_enable_x64", True)

# Plotting
import seaborn as sn
import matplotlib.pyplot as plt
cmap = sn.color_palette("viridis", as_cmap=True)

# JAXVacua
import jaxvacua as jvc

warnings.filterwarnings('ignore')

## Calling conventions

JAXVacua accepts two equivalent interfaces for custom geometries, via the constructor arguments `prepotential_input` and `period_input`.

---

### Option A: Prepotential input — `prepotential_input=F`

```python
def F(X, conj=False):
    ...
    return complex_scalar   # JAX complex scalar
```

| Argument | Type | Description |
|----------|------|-------------|
| `X` | `jnp.ndarray`, shape `(h12+1,)` | Homogeneous coordinates $(X^0, X^1, \ldots, X^{h^{1,2}})$ |
| `conj` | `bool` | When `True`, return $\bar{F}(\bar{X})$ (conjugate prepotential). Needed for the Kähler potential. |
| **Return** | complex scalar | The prepotential $F(X)$ (or its conjugate when `conj=True`) |

**`conj` flag guidance:**
- If your prepotential has **only real-valued parameters** (e.g. $F = X^1{}^3/X^0$), you can **ignore the flag** — JAXVacua applies `jnp.conj` automatically.
- If it involves **complex constants** (e.g. the K-point prepotential below), you **must implement both branches** of `if conj:` explicitly so that `F(X, conj=True) == jnp.conj(F(jnp.conj(X)))`.

The period vector is derived automatically via $\Pi = (\tilde{F}_0,\ \partial_1 F,\ \ldots,\ X^0,\ X^1,\ \ldots)^T$, where $\tilde{F}_0 = 2F - X^a\partial_a F$.

---

### Option B: Period vector input — `period_input=Pi`

```python
def Pi(X):
    ...
    return jnp.array([...])   # shape (2*(h12+1),)
```

| Argument | Type | Description |
|----------|------|-------------|
| `X` | `jnp.ndarray`, shape `(h12+1,)` | Homogeneous coordinates |
| **Return** | `jnp.ndarray`, shape `(2*(h12+1),)` | Full period vector $\Pi = (\tilde{F}_0,\ \partial_1 F,\ \ldots,\ X^0,\ \ldots)^T$ |

---

### JAX compatibility

Both `F` and `Pi` must be **fully JAX-traceable**: no Python-level branching on traced values, no NumPy calls. Use `jax.lax.cond` for data-dependent branches. Static parameters (scalars, model constants) can be captured via closures or `@partial(jit, static_argnums=...)`.

### Symplectic normalisation

The Kähler potential is $K = -\log(i\,\Pi^\dagger \Sigma\, \Pi)$, where $\Sigma$ is the symplectic matrix (accessible as `model.periods.sigma()`). A valid model requires $i\,\Pi^\dagger \Sigma\, \Pi > 0$ throughout the interior of moduli space. The overall scale of $\Pi$ affects $K$ but not the SUSY vacuum equations.

* **Test 1 modulus example e.g. from Damian/Lorenz**? 
* **Test example from Thomas?**

## Testing prepotential input

In [ ]:
def F(X,conj=False):
    
    return X[1]**3/X[0]

In [ ]:
F(jnp.ones(2)*1j)

In [ ]:
model = jvc.FluxEFT(h12=1,prepotential_input=F,limit=None)
model

In [ ]:
model.F(jnp.ones(1)*1j)

In [ ]:
flux = np.array([ 9, 10, -6,  7, -1,  1,  0,  6])
z0 = jnp.ones(1)*1j
tau = 10*1j
model.DW(z0,jnp.conj(z0),tau,jnp.conj(tau),flux)

## Testing period input

In [ ]:
def per(X,conj=False):
    
    return X[0]*jnp.array([-X[1]**3/X[0]**2,3*X[1]**2/X[0],X[0],X[1]])

In [ ]:
model = jvc.FluxEFT(h12=1,limit=None,model_type=None,period_input=per)
model

In [ ]:
flux = np.array([ 9, 10, -6,  7, -1,  1,  0,  6])
z0 = jnp.ones(1)*1j
tau = 10*1j
model.DW(z0,jnp.conj(z0),tau,jnp.conj(tau),flux)

This is the same output as above for the prepotential input (as it should be).

### Symplectic pairing validation

We verify that $i\,\Pi^\dagger \Sigma\, \Pi > 0$ at a test point. This quantity equals $e^{-K}$, so it must be a positive real number for the Kähler potential to be well-defined.

In [ ]:
# Test at z = 1.5i (a point in the upper half-plane)
z_test = jnp.array([1.5j])
X_test = jnp.array([1., z_test[0]])

Pi_eval = per(X_test)
sigma   = model.periods.sigma()

# i * Pi^dagger @ sigma @ Pi  should be real and positive (= e^{-K})
pairing = 1j * (Pi_eval.conj() @ sigma @ Pi_eval)
K_model = float(model.kahler_potential(z_test, jnp.conj(z_test)).real)

print(f"i Π† Σ Π  = {pairing:.6f}")
print(f"  real part = {pairing.real:.6f}  (> 0 ✓)")
print(f"  imag part = {pairing.imag:.2e}  (should be ≈ 0)")
print()
print(f"-log(i Π† Σ Π) = {-np.log(float(pairing.real)):.6f}")
print(f"model.kahler_potential(z) = {K_model:.6f}")

## Explicit example

Let us take a concrete examples
[2306.01059](https://arxiv.org/abs/2306.01059)

In [ ]:
@partial(jit,static_argnums=(1,2,3,4,5,6,7,8,))
def prepot_Kpoint(X,delta=0,tau=0,gamma=0,c=0,B1=0,B2=0,B3=0,conj=False):
    
    tau2 = tau.imag
    
    if conj:
        tau = jnp.conj(tau)
    
    s = (X[1]-tau)/X[0]
    
    F0 = delta/2+gamma*tau
    
    if conj:
        F1 = gamma-c*tau2/2/jnp.pi*(jnp.log(-1j*s/2/B1/tau2)-1)
        F2 = -1j*c/4/jnp.pi**2*(jnp.log(-1j*s/2/B1/tau2)-2+B2/4/B1**2)
    else:
        F1 = gamma-c*tau2/2/jnp.pi*(jnp.log(1j*s/2/B1/tau2)-1)
        F2 = 1j*c/4/jnp.pi**2*(jnp.log(1j*s/2/B1/tau2)-2+B2/4/B1**2)
    
    F3 = c/16/jnp.pi/tau2*(1-B2**2/B1**4+2*B3/3/B1*+3)
    
    return F0+s*F1+(s**2)*F2+(s**3)*F3


@partial(jit,static_argnums=(1,2,3,4,5,6,7,8,))
def prepot_Cpoint(X,delta=0,tau=0,gamma=0,k=0,A1=1,A2=1,conj=False):
    
    tau2 = tau.imag
    
    if conj:
        tau = jnp.conj(tau)
    
    s = (X[1]-tau)/X[0]
    
    F0 = tau/2
    F1 = delta-gamma*tau
    
    if conj:
        F2 = -1j*k*(3-2*jnp.log(s)+2*jnp.log(A1)) -gamma*F1
        F3 = 1j*k*(3*A1**2*gamma-A2)/12/jnp.pi/A1**2
    else:
        F2 = 1j*k*(3-2*jnp.log(s)+2*jnp.log(A1)) -gamma*F1
        F3 = -1j*k*(3*A1**2*gamma-A2)/12/jnp.pi/A1**2
    
    return F0+s*F1+(s**2)*F2+(s**3)*F3

### K-point $X_{3,3}$

In [ ]:
tau = -1/2+1j*np.sqrt(3)/2
gamma = 1/6
delta = 1/3
B1 = 1
B2 = 0
B3 = 0
c= 2

import scipy
LF1 = scipy.special.gamma(1/3)**6/8/np.sqrt(3)/np.pi**3

coeff_norm = -1/3/LF1**2

@partial(jit,static_argnums=(1,))
def F(X,conj=False):
    
    return prepot_Kpoint(X,delta=delta,tau=tau,gamma=gamma,c=c,B1=coeff_norm*B1,B2=coeff_norm*B2,B3=coeff_norm*B3,conj=conj)

In [ ]:
F(jnp.ones(2)*1j)

In [ ]:
F(-jnp.ones(2)*1j,conj=True)

In [ ]:
model = jvc.FluxEFT(h12=1,limit=None,model_type=None,prepotential_input=F)
model

In [ ]:
model.F(jnp.ones(1)*1j)

In [ ]:
flux = np.array([ 9, 10, -6,  7, -1,  1,  0,  6])
z0 = jnp.ones(1)*1j
tau = 10*1j
model.DW(z0,jnp.conj(z0),tau,jnp.conj(tau),flux)

In [ ]:
# Compute period vector
def Pi(X):
    F_0 = -X[1]**3/X[0]
    F_1 = 3*X[1]**2
    return jnp.array([F_0,F_1,X[0],X[1]])

# Construct model
model = jvc.FluxEFT(h12=1,limit="'String Data 2025'",period_input=Pi)
model

In [ ]:
# Compute prepotential
def F(X,conj=False):
    return -X[1]**3/X[0]
    
# Construct model
model = jvc.FluxEFT(h12=1,limit="'String Data 2025'",prepotential_input=F)
model

## Vacuum search with a custom prepotential

Once a `FluxEFT` model is built from `prepotential_input` or `period_input`, the full JAXVacua vacuum-finding machinery applies without modification. Below we run a complete search using the simple cubic prepotential $F = X^1{}^3/X^0$ (equivalent to `limit="LCS"` at $h^{1,2}=1$), which is the cleanest self-contained example.

The same code works unchanged for the K-point prepotential defined above — simply swap the `F` function.

In [ ]:
from jax import vmap
from jaxvacua.util import vmapping_func

# --- define the cubic prepotential ---
def F_cubic(X, conj=False):
    return X[1]**3 / X[0]

# --- build model (FluxVacuaFinder inherits FluxEFT and adds the sampler) ---
model_vf = jvc.FluxVacuaFinder(h12=1, prepotential_input=F_cubic, limit=None, Q=40)
sampler  = jvc.data_sampler(model_vf,
                             flux_bounds=[-4, 4],
                             moduli_bounds=[1, 5],
                             axion_bounds=[-0.5, 0.5],
                             dilaton_bounds=[2, 10])

print(model_vf)
print(f"D3-tadpole bound Q = {model_vf.D3_tadpole}")

In [ ]:
# --- define vmapped objective and optimiser ---
DW_v   = vmap(model_vf.DW)
kwargs = {"Q": model_vf.D3_tadpole, "return_flag": True,
          "constraints": None, "remove_NANs": True, "in_axes": (0, 0, 0)}
optimiser = vmapping_func(model_vf.linearised_shifts, mode="Fflux", **kwargs)

# --- run sampling loop ---
seed    = 42
rns_key = jvc.PRNGSequence(seed)

moduli, tau, fluxes, residuals = model_vf.sample_SUSY_flux_vacua(
    N=100,
    rns_key=rns_key,
    sampler=sampler,
    optimiser=optimiser,
    objective_fct=DW_v,
    vmap_dim=50,
    print_progress=True,
)
print(f"\nFound {len(moduli)} SUSY vacua")

In [ ]:
# --- inspect and validate vacua ---
if len(moduli) > 0:
    W0v      = vmap(model_vf.superpotential_gauge_invariant)
    tadpole_v = vmap(model_vf.tadpole)
    DW_check  = vmap(model_vf.DW)

    W0     = W0v(moduli, tau, fluxes)
    Nflux  = tadpole_v(fluxes)
    DW_res = jnp.sum(jnp.abs(DW_check(moduli, jnp.conj(moduli), tau, jnp.conj(tau), fluxes)), axis=1)

    print(f"{'z':>25}  {'τ':>22}  {'|W₀|':>10}  {'gₛ':>7}  {'N_flux':>7}  {'|DW|':>10}")
    print("-" * 90)
    for i in range(min(8, len(moduli))):
        z_i  = moduli[i][0]
        t_i  = tau[i]
        print(f"  {z_i.real:+.4f}{z_i.imag:+.4f}j  "
              f"{t_i.real:+.4f}{t_i.imag:+.4f}j  "
              f"{abs(W0[i]):.4e}  "
              f"{(1/t_i.imag):.4f}  "
              f"{int(Nflux[i]):>7}  "
              f"{float(DW_res[i]):.2e}")
    if len(moduli) > 8:
        print(f"  ... ({len(moduli) - 8} more)")
else:
    print("No vacua found. Try increasing N or widening sampler bounds.")

## Summary

This notebook showed how to feed custom geometry data into JAXVacua. Key takeaways:

- Use `prepotential_input=F` when you know $F(X)$ analytically. JAXVacua derives the period vector automatically.
- Use `period_input=Pi` when you prefer to supply $\Pi(X)$ directly (equivalent output).
- The `conj` flag is only needed if the prepotential contains explicit complex constants; for purely real-parameter prepotentials it can be ignored.
- Validate your input with the symplectic pairing: $i\,\Pi^\dagger \Sigma\, \Pi > 0$ must hold at every point of moduli space.
- Once the model is built, the full vacuum-finding workflow (`FluxVacuaFinder`, `data_sampler`, `sample_SUSY_flux_vacua`) works without modification.

**Next steps:**
- [05_finding_flux_vacua](../02_vacuum_finding/05_finding_flux_vacua.ipynb) — detailed Newton refinement workflow
- [15_hypergeometric_models](15_hypergeometric_models.ipynb) — built-in one-modulus prepotentials (LCS and K-point) with closed-form analytic results
- [8_ISD_sampling_wrapper](../02_vacuum_finding/8_ISD_sampling_wrapper.ipynb) — large-scale sampling using the same `FluxVacuaFinder` interface